# Brawl Stars Brawler Classification
4 classes: Bull, Leon, Shelly, Spike. 20 original screenshots per class + 180 augmented copies (`augment.py`).

## 1. Data loading and split
The **originals** are split 12 / 4 / 4. Copies go to the same part as their original, so one screenshot never ends up in both train and test.

In [1]:
import random

DATA = 'dataset'
CLASSES = ['bull', 'leon', 'shelly', 'spike']

random.seed(42)
train, val, test = [], [], []

for label in range(4):
    name = CLASSES[label]
    numbers = list(range(1, 21))
    random.shuffle(numbers)

    for i in numbers[:12]:
        train.append((f'{DATA}/{name}/{name}_{i:03d}.jpg', label))
        first = 21 + (i - 1) * 9
        for n in range(first, first + 9):
            train.append((f'{DATA}/{name}/{name}_{n:03d}.jpg', label))

    for i in numbers[12:16]:
        val.append((f'{DATA}/{name}/{name}_{i:03d}.jpg', label))
    for i in numbers[16:]:
        test.append((f'{DATA}/{name}/{name}_{i:03d}.jpg', label))

print('train:', len(train), 'val:', len(val), 'test:', len(test))

train: 480 val: 16 test: 16


## 2. Normalization and PyTorch preparation
Image → 224×224 → tensor (values 0–1) → normalization with ImageNet mean and std.
DataLoader gives the images in batches of 32.

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from torchvision import transforms
from PIL import Image

prepare = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def to_dataset(items):
    images = []
    labels = []
    for path, label in items:
        img = Image.open(path).convert('RGB')
        images.append(prepare(img))
        labels.append(label)
    return TensorDataset(torch.stack(images), torch.tensor(labels))

train_loader = DataLoader(to_dataset(train), batch_size=32, shuffle=True)
val_loader = DataLoader(to_dataset(val), batch_size=32)
test_loader = DataLoader(to_dataset(test), batch_size=32)

images, labels = next(iter(train_loader))
print('batch shape:', images.shape)
print('min:', images.min().item(), 'max:', images.max().item())

## 3. Baseline model: simple CNN
Three blocks (convolution → ReLU → max pooling) and one linear layer with 4 outputs.

In [ ]:
from torch import nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

def make_cnn():
    torch.manual_seed(42)
    model = nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(64 * 28 * 28, 4),
    )
    return model.to(device)

simple_cnn = make_cnn()
out = simple_cnn(images.to(device))
print('Here is simple CNN, output shape:', out.shape)
print('parameters:', sum(p.numel() for p in simple_cnn.parameters()))

## 4. Training loop
One epoch = the model sees all 480 training images once. After each epoch we check accuracy on validation.
The same function is used for ResNet18.

In [ ]:
import matplotlib.pyplot as plt

def accuracy(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            out = model(images.to(device))
            correct += (out.argmax(1).cpu() == labels).sum().item()
    return correct / len(loader.dataset)

def train_model(model, train_loader, val_loader, epochs, lr, show=True):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses, train_accs, val_accs = [], [], []

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            out = model(images)
            loss = loss_fn(out, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)

        losses.append(total_loss / len(train_loader.dataset))
        train_accs.append(accuracy(model, train_loader))
        val_accs.append(accuracy(model, val_loader))
        if show:
            print(f'epoch {epoch + 1}: loss {losses[-1]:.3f}, train {train_accs[-1]:.2f}, val {val_accs[-1]:.2f}')

    return losses, train_accs, val_accs

def plot_history(losses, train_accs, val_accs, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    ax1.plot(losses)
    ax1.set_title(f'{title}: loss')
    ax1.set_xlabel('epoch')
    ax2.plot(train_accs, label='train')
    ax2.plot(val_accs, label='val')
    ax2.set_title(f'{title}: accuracy')
    ax2.set_xlabel('epoch')
    ax2.set_ylim(0, 1.05)
    ax2.legend()
    plt.show()

cnn_history = train_model(simple_cnn, train_loader, val_loader, epochs=15, lr=0.001)
plot_history(*cnn_history, 'Simple CNN')

## 5. Improved model: transfer learning (ResNet18)
ResNet18 pretrained on ImageNet (1.2M images). Only the last layer is replaced: 4 classes instead of 1000.
The whole network is fine-tuned with a small learning rate (0.0001).

In [ ]:
from torchvision import models

def make_resnet():
    torch.manual_seed(42)
    model = models.resnet18(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(512, 4)
    return model.to(device)

resnet = make_resnet()
print('Here is ResNet18, parameters:', sum(p.numel() for p in resnet.parameters()))

resnet_history = train_model(resnet, train_loader, val_loader, epochs=10, lr=0.0001)
plot_history(*resnet_history, 'ResNet18')

## 6. Test evaluation
Test = 16 originals (4 per class) that the models never saw.
- **accuracy** — share of correct answers;
- **precision** — of all images predicted as a class, how many really are that class;
- **recall** — of all real images of a class, how many the model found;
- **F1** — balance of precision and recall;
- **confusion matrix** — row = true class, column = predicted class. Everything off the diagonal is a mistake.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def predict(model, loader):
    model.eval()
    true, preds = [], []
    with torch.no_grad():
        for images, labels in loader:
            out = model(images.to(device))
            preds += out.argmax(1).cpu().tolist()
            true += labels.tolist()
    return true, preds

for model, title in [(simple_cnn, 'Simple CNN'), (resnet, 'ResNet18')]:
    true, preds = predict(model, test_loader)
    print(title)
    print(classification_report(true, preds, target_names=CLASSES, digits=3, zero_division=0))
    ConfusionMatrixDisplay(confusion_matrix(true, preds), display_labels=CLASSES).plot(cmap='Blues')
    plt.title(title)
    plt.show()

## 7. Cross-validation (5 folds)
A test of 16 images is too small: one mistake = 6%. The 20 originals of each class are split into 5 groups of 4.
5 times: one group is the test, the other 16 originals + their copies are the training set. Both models are trained from scratch.
Every one of the 80 originals is tested exactly once; the results are averaged.

In [ ]:
import numpy as np

random.seed(42)
order = {}
for name in CLASSES:
    numbers = list(range(1, 21))
    random.shuffle(numbers)
    order[name] = numbers

cnn_scores, resnet_scores = [], []
mistakes = []

for k in range(5):
    cv_train, cv_test = [], []
    for label in range(4):
        name = CLASSES[label]
        for i in order[name]:
            if i in order[name][k * 4:k * 4 + 4]:
                cv_test.append((f'{DATA}/{name}/{name}_{i:03d}.jpg', label))
            else:
                cv_train.append((f'{DATA}/{name}/{name}_{i:03d}.jpg', label))
                first = 21 + (i - 1) * 9
                for n in range(first, first + 9):
                    cv_train.append((f'{DATA}/{name}/{name}_{n:03d}.jpg', label))
    cv_train_loader = DataLoader(to_dataset(cv_train), batch_size=32, shuffle=True)
    cv_test_loader = DataLoader(to_dataset(cv_test), batch_size=32)

    cnn = make_cnn()
    train_model(cnn, cv_train_loader, cv_test_loader, epochs=15, lr=0.001, show=False)
    cnn_scores.append(accuracy(cnn, cv_test_loader))

    res = make_resnet()
    train_model(res, cv_train_loader, cv_test_loader, epochs=10, lr=0.0001, show=False)
    resnet_scores.append(accuracy(res, cv_test_loader))

    true, preds = predict(res, cv_test_loader)
    for j in range(len(cv_test)):
        if preds[j] != true[j]:
            mistakes.append((cv_test[j][0], CLASSES[preds[j]]))

    print(f'fold {k + 1}: simple CNN {cnn_scores[-1]:.3f}, ResNet18 {resnet_scores[-1]:.3f}')

print(f'Simple CNN: {np.mean(cnn_scores):.3f} ± {np.std(cnn_scores):.3f}')
print(f'ResNet18:   {np.mean(resnet_scores):.3f} ± {np.std(resnet_scores):.3f}')

## 8. Error analysis
All images that ResNet18 got wrong during cross-validation (title: true class -> predicted class).

In [ ]:
print('ResNet18 mistakes:', len(mistakes), 'of 80')

plt.figure(figsize=(15, 3.5))
for j in range(len(mistakes)):
    path, pred = mistakes[j]
    true_name = path.split('/')[1]
    file_name = path.split('/')[2]
    plt.subplot(1, len(mistakes), j + 1)
    plt.imshow(Image.open(path))
    plt.title(f'{true_name} -> {pred}\n{file_name}')
    plt.axis('off')
plt.show()

### Why the model made these mistakes
- **4 of 5 mistakes are skins that were missing from the training set of that fold.**
  `bull_004` was tested together with `bull_001` (the same default skin), `leon_019` together with `leon_006` (the same white hood skin),
  `spike_020` together with `spike_019` (the same coffin skin), and the skin of `bull_020` appears only once in the whole dataset.
  So the model learns skins it has already seen better than the brawler itself.
- **1 mistake is a bad frame:** in `spike_004` the character is almost out of the frame.
- **Colors and background matter:** Bull in white and pink on a purple background was predicted as Shelly, whose skins are often pink and purple.
  Spike in a purple and green coffin skin was predicted as Leon.

### Conclusion
| Model | Test accuracy (16 images) | Cross-validation accuracy (80 images) |
|---|---|---|
| Simple CNN | 0.812 | 0.800 ± 0.061 |
| ResNet18 | 0.875 | 0.938 ± 0.040 |

Transfer learning (ResNet18) is better than the simple CNN on every fold. The simple CNN overfits: 100% on train, about 75–80% on new images.
To improve further: more screenshots with more different skins and back views.